In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.stats import shapiro, probplot
import seaborn as sns
import scipy.stats as stats
from datetime import datetime
from django_pandas.io import read_frame
from pathlib import Path
from datetime import timedelta


from dj_notebook import activate

# pd.options.mode.copy_on_write = True
# pd.options.mode.chained_assignment = "raise"
env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)

In [ ]:
from edc_model_to_dataframe import read_frame_edc
from edc_appointment.models import Appointment
from intecomm_subject.models import SubjectVisit, SubjectVisitMissed, DrugRefillHiv
from intecomm_analytics.dataframes import get_df_main_1858

In [ ]:
df_main = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_appt= read_frame(Appointment.objects.all())
df_appt[df_appt.visit_code_sequence==0].appt_status.value_counts()

In [ ]:
df_visit= read_frame_edc(SubjectVisit.objects.all())
df_visit.rename(columns={'id':'subject_visit_id'}, inplace=True)
df_missed= read_frame_edc(SubjectVisitMissed.objects.all())
df_missed.rename(columns={"report_datetime": "missed_visit_report_datetime"}, inplace=True)
df_visit = df_visit.merge(df_missed[["subject_visit_id", "missed_visit_report_datetime"]], on="subject_visit_id", how="left")

In [ ]:
df_visit = df_visit.merge(df_main[["subject_identifier", "assignment", "htn", "hiv", "hiv_only", "dm"]], on="subject_identifier", how="left")

In [ ]:
df_hiv_refill = read_frame_edc(DrugRefillHiv.objects.all())
# df_hiv_refill[df_hiv_refill.subject_identifier=="107-209-0010-8"].sort_values(by=["visit_code"])[["subject_identifier", "rx_days", "visit_code", "visit_datetime"]]
df = df_hiv_refill.copy()
df = df.merge(df_main[["subject_identifier", "assignment"]], on="subject_identifier", how="left")
df.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df.reset_index(drop=True, inplace=True)


df["next_visit_datetime"] = df.groupby("subject_identifier")["visit_datetime"].shift(-1)
df["interval_check"] = (df["visit_datetime"] + pd.to_timedelta(df["rx_days"], unit='d') + timedelta(days=3)) >= df["next_visit_datetime"]
df["interval_days"] = df["next_visit_datetime"] - (df["visit_datetime"] + pd.to_timedelta(df["rx_days"], unit='d'))
df["interval_days"] = df.apply(lambda row: timedelta(days=0) if row.interval_days <= timedelta(days=0) else row.interval_days, axis=1)

In [ ]:
df = df[["subject_identifier", "visit_datetime", "assignment", "interval_days"]].copy()
df.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df.reset_index(drop=True, inplace=True)
df["interval_days"] = df["interval_days"].apply(lambda x: x.days)

In [ ]:
print(df.head())

In [ ]:
df["interval_days"].describe()

In [ ]:
# sum days missed medication by subject
df_a = df[df["assignment"]=="a"].groupby("subject_identifier")["interval_days"].sum()
df_b = df[df["assignment"]=="b"].groupby("subject_identifier")["interval_days"].sum()

In [ ]:
df_a.describe()

In [ ]:
df_b.describe()

In [ ]:
print("Total interval days per person in group A:")
print(df_a)
print("\nTotal interval days per person in group B:")
print(df_b)

In [ ]:
# Plot histograms and Q-Q plots to visually inspect the distribution
plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
sns.histplot(df_a, kde=True)
plt.title('Histogram of Interval Days (Group A)')

plt.subplot(2, 2, 2)
sns.histplot(df_b, kde=True)
plt.title('Histogram of Interval Days (Group B)')

# Plot Q-Q plots to visually inspect the distribution
plt.subplot(2, 2, 3)
probplot(df_a, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group A)')

plt.subplot(2, 2, 4)
probplot(df_b, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group B)')

plt.tight_layout()
plt.show()

In [ ]:
# Perform the Shapiro-Wilk test for normality
shapiro_a_stat, shapiro_a_p_value = shapiro(df_a)
shapiro_b_stat, shapiro_b_p_value = shapiro(df_b)

# Display the results of the Shapiro-Wilk test
print(f"Shapiro-Wilk Test for Group A: Statistic={shapiro_a_stat}, P-value={shapiro_a_p_value}")
print(f"Shapiro-Wilk Test for Group B: Statistic={shapiro_b_stat}, P-value={shapiro_b_p_value}")

Interpretation:

Histograms: The histograms show the distribution of interval days for each group. If the data follows a normal distribution, the histogram should resemble a bell curve.

Q-Q Plots: The Q-Q plots compare the quantiles of the data to the quantiles of a normal distribution. If the data follows a normal distribution, the points should lie along the diagonal line.

Shapiro-Wilk Test: The Shapiro-Wilk test checks for normality. A low p-value (typically less than 0.05) indicates that the data does not follow a normal distribution.
In this case, the p-values for both groups are extremely low, indicating that the data does not follow a normal distribution.

In [ ]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(df_a.values.astype(float), df_b.values.astype(float))
print(f"U-statistic: {u_stat}, P-value: {p_value}")

print("The Mann-Whitney U test revealed a statistically significant difference in the interval "
f"days per person between Group A and Group B (U = {u_stat}, p = {p_value}), "
"indicating that the distributions of interval days differ between the two groups.")


In [ ]:
from intecomm_subject.models import DrugRefillDm

df_dm_refill = read_frame_edc(DrugRefillDm.objects.all())
df_dm = df_dm_refill.copy()
df_dm = df_dm.merge(df_main[["subject_identifier", "assignment"]], on="subject_identifier", how="left")
df_dm.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df_dm.reset_index(drop=True, inplace=True)

df_dm["next_visit_datetime"] = df_dm.groupby("subject_identifier")["visit_datetime"].shift(-1)
df_dm["interval_check"] = (df_dm["visit_datetime"] + pd.to_timedelta(df_dm["rx_days"], unit='d') + timedelta(days=3)) >= df_dm["next_visit_datetime"]
df_dm["interval_days"] = df_dm["next_visit_datetime"] - (df_dm["visit_datetime"] + pd.to_timedelta(df_dm["rx_days"], unit='d'))
df_dm["interval_days"] = df_dm.apply(lambda row: timedelta(days=0) if row.interval_days <= timedelta(days=0) else row.interval_days, axis=1)

df_dm = df_dm[["subject_identifier", "visit_datetime", "assignment", "interval_days"]].copy()
df_dm.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df_dm.reset_index(drop=True, inplace=True)
df_dm["interval_days"] = df_dm["interval_days"].apply(lambda x: x.days)




In [ ]:
df_dm

In [ ]:
df_dm_a = df_dm[df_dm["assignment"]=="a"].groupby("subject_identifier")["interval_days"].sum()
df_dm_b = df_dm[df_dm["assignment"]=="b"].groupby("subject_identifier")["interval_days"].sum()

In [ ]:
df_dm_a.describe()

In [ ]:
df_dm_b.describe()


In [ ]:
print("Total interval days per person in group A:")
print(df_dm_a)
print("\nTotal interval days per person in group B:")
print(df_dm_b)

In [ ]:
# Plot histograms and Q-Q plots to visually inspect the distribution
plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
sns.histplot(df_dm_a, kde=True)
plt.title('Histogram of Interval Days (Group A)')

plt.subplot(2, 2, 2)
sns.histplot(df_dm_b, kde=True)
plt.title('Histogram of Interval Days (Group B)')

# Plot Q-Q plots to visually inspect the distribution
plt.subplot(2, 2, 3)
probplot(df_dm_a, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group A)')

plt.subplot(2, 2, 4)
probplot(df_dm_b, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group B)')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(df_dm_a.values.astype(float), df_dm_b.values.astype(float))
print(f"U-statistic: {u_stat}, P-value: {p_value}")

print("The Mann-Whitney U test revealed a statistically significant difference in the interval "
f"days per person between Group A and Group B (U = {u_stat}, p = {p_value}), "
"indicating that the distributions of interval days differ between the two groups.")


In [ ]:
from intecomm_subject.models import DrugRefillHtn

df_htn_refill = read_frame_edc(DrugRefillHtn.objects.all())
df_htn = df_htn_refill.copy()
df_htn = df_htn.merge(df_main[["subject_identifier", "assignment"]], on="subject_identifier", how="left")
df_htn.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df_htn.reset_index(drop=True, inplace=True)

df_htn["next_visit_datetime"] = df_htn.groupby("subject_identifier")["visit_datetime"].shift(-1)
df_htn["interval_check"] = (df_htn["visit_datetime"] + pd.to_timedelta(df_htn["rx_days"], unit='d') + timedelta(days=3)) >= df_htn["next_visit_datetime"]
df_htn["interval_days"] = df_htn["next_visit_datetime"] - (df_htn["visit_datetime"] + pd.to_timedelta(df_htn["rx_days"], unit='d'))
df_htn["interval_days"] = df_htn.apply(lambda row: timedelta(days=0) if row.interval_days <= timedelta(days=0) else row.interval_days, axis=1)

df_htn = df_htn[["subject_identifier", "visit_datetime", "assignment", "interval_days"]].copy()
df_htn.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df_htn.reset_index(drop=True, inplace=True)
df_htn["interval_days"] = df_htn["interval_days"].apply(lambda x: x.days)


In [ ]:
df_htn

In [ ]:
df_htn_a = df_htn[df_htn["assignment"]=="a"].groupby("subject_identifier")["interval_days"].sum()
df_htn_b = df_htn[df_htn["assignment"]=="b"].groupby("subject_identifier")["interval_days"].sum()

In [ ]:
df_htn_a.describe()

In [ ]:
df_htn_b.describe()

In [ ]:
print("Total interval days per person in group A:")
print(df_htn_a)
print("\nTotal interval days per person in group B:")
print(df_htn_b)

In [ ]:
# Plot histograms and Q-Q plots to visually inspect the distribution
plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
sns.histplot(df_htn_a, kde=True)
plt.title('Histogram of Interval Days (Group A)')

plt.subplot(2, 2, 2)
sns.histplot(df_htn_b, kde=True)
plt.title('Histogram of Interval Days (Group B)')

# Plot Q-Q plots to visually inspect the distribution
plt.subplot(2, 2, 3)
probplot(df_htn_a, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group A)')

plt.subplot(2, 2, 4)
probplot(df_htn_b, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group B)')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(df_htn_a.values.astype(float), df_htn_b.values.astype(float))
print(f"U-statistic: {u_stat}, P-value: {p_value}")

print("The Mann-Whitney U test revealed a statistically significant difference in the interval "
f"days per person between Group A and Group B (U = {u_stat}, p = {p_value}), "
"indicating that the distributions of interval days differ between the two groups.")